# Thermal + Radar teacher→student training

This notebook runs the versioned training code exported beside the NPZ shards. Unknown labels are masked; train/validation remain split by complete session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/thermal-fusion/gexport/v2'
STUDENT = 'both'       # thermal, radar, or both
EPOCHS = 50
BATCH_SIZE = 32
WORKERS = 2

In [ ]:
import json, os, sys, torch
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU'
for required in ('manifest.json', 'code/perception/train_students.py'):
    assert os.path.exists(os.path.join(DATA, required)), f'missing {required}; re-export/upload current bundle'
with open(os.path.join(DATA, 'manifest.json')) as f:
    manifest = json.load(f)
print('GPU:', torch.cuda.get_device_name(0))
print('split:', manifest['split'])

In [ ]:
import subprocess
env = dict(os.environ)
env['PYTHONPATH'] = os.path.join(DATA, 'code')
cmd = [sys.executable, '-m', 'perception.train_students',
       '--data', DATA, '--student', STUDENT, '--epochs', str(EPOCHS),
       '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS)]
print(' '.join(cmd))
subprocess.run(cmd, env=env, check=True)

In [ ]:
import glob
print('checkpoints:')
for path in sorted(glob.glob(os.path.join(DATA, 'models', '*.pt'))):
    print(' ', path)